In [3]:
# =============================================================================
# [FILE 2] miryang_report_agent.py
# 밀양시 교통사고 위험 — GPT 기반 자동 보고서 생성 에이전트
#
# 선행 조건 : 1_miryang_analysis.py 실행 완료
#             (TableVI_LLM_Dataset.csv 가 동일 폴더에 있어야 함)
#
# API 키 설정 방법 (둘 중 하나 선택):
#   방법 1 — .env 파일 생성:
#             OPENAI_API_KEY=sk-xxxxxxxxxxxxxxxxxxxxxxxx
#             LLM_MODEL=gpt-4o-mini
#   방법 2 — 아래 직접할당 주석 해제
#
# 설치 : pip install openai python-dotenv
# =============================================================================

import os
import json
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

# =============================================================================
# 0. API 키 설정
# =============================================================================

load_dotenv()

# ── 방법 2: 직접 할당 (테스트용) ─────────────────────────────────────────
# os.environ['OPENAI_API_KEY'] = 'sk-xxxxxxxxxxxxxxxxxxxxxxxx'
# os.environ['LLM_MODEL']      = 'gpt-4o-mini'

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL', 'gpt-4o-mini')

if not OPENAI_API_KEY:
    raise ValueError(
        "\n[ERROR] OPENAI_API_KEY가 설정되지 않았습니다.\n"
        "  방법 1: 프로젝트 폴더에 .env 파일 생성 후\n"
        "          OPENAI_API_KEY=sk-... 입력\n"
        "  방법 2: 이 파일 상단 직접할당 주석 해제"
    )
print(f"[INFO] Model: {LLM_MODEL}")

# =============================================================================
# 1. 피처명 영문 서술어 매핑 (프롬프트용)
# =============================================================================

FEATURE_DESC = {
    'viol_unsafe_driving'      : 'Unsafe Driving Violations',
    'viol_pedestrian_prot'     : 'Pedestrian Protection Violations',
    'viol_signal'              : 'Signal Violations',
    'viol_centerline'          : 'Centerline Crossing Violations',
    'viol_intersection_method' : 'Intersection Method Violations',
    'viol_unsafe_distance'     : 'Unsafe Following Distance Violations',
    'viol_speeding'            : 'Speeding Violations',
    'viol_overtaking'          : 'Illegal Overtaking',
    'viol_lane'                : 'Lane Violations',
    'viol_railroad'            : 'Railroad Crossing Violations',
    'road_dry'                 : 'Dry Road Surface Incidents',
    'road_wet'                 : 'Wet/Humid Road Surface Incidents',
    'road_frost'               : 'Frost/Ice Road Incidents',
    'road_snow'                : 'Snow-Covered Road Incidents',
    'weather_clear'            : 'Clear Weather Incidents',
    'weather_cloudy'           : 'Cloudy Weather Incidents',
    'weather_rain'             : 'Rainy Weather Incidents',
    'weather_fog'              : 'Foggy Weather Incidents',
    'road_single_other'        : 'Single Road (Other) Incidents',
    'road_intersection'        : 'Intersection Incidents',
    'road_near_intersection'   : 'Near-Intersection Incidents',
    'sports_facilities'        : 'Number of Sports Facilities',
    'campgrounds'              : 'Number of Campgrounds',
    'population'               : 'Resident Population',
    'industrial_facilities'    : 'Number of Industrial Facilities',
    'tourist_spots'            : 'Number of Tourist Spots',
    'transport_companies'      : 'Number of Transport Companies',
    'local_events'             : 'Number of Local Events',
    # 한국어 원본 fallback
    '법규위반_안전운전불이행_건수'    : 'Unsafe Driving Violations',
    '법규위반_보행자보호의무위반_건수' : 'Pedestrian Protection Violations',
    '법규위반_신호위반_건수'          : 'Signal Violations',
    '노면상태_건조_건수'              : 'Dry Road Surface Incidents',
    '노면상태_젖음습기_건수'          : 'Wet Road Surface Incidents',
    '체육시설_개수'                   : 'Number of Sports Facilities',
    '인구수'                          : 'Resident Population',
}

def feat_to_desc(name: str) -> str:
    return FEATURE_DESC.get(name, name.replace('_',' ').title())

# =============================================================================
# 2. 데이터 로드
# =============================================================================
print("\n" + "=" * 60)
print("STEP 1 | Loading LLM Dataset")
print("=" * 60)

LLM_PATH  = './TableVI_LLM_Dataset.csv'
DICE_PATH = './TableV_DiCE_Policy.csv'

try:
    df_llm = pd.read_csv(LLM_PATH)
    print(f"  Loaded: {LLM_PATH}  shape={df_llm.shape}")
except FileNotFoundError:
    raise FileNotFoundError(
        f"[ERROR] {LLM_PATH} 없음.\n"
        "  1_miryang_analysis.py 를 먼저 실행하세요."
    )

DICE_AVAILABLE = False
df_dice = pd.DataFrame()
try:
    df_dice = pd.read_csv(DICE_PATH)
    DICE_AVAILABLE = True
    print(f"  Loaded: {DICE_PATH}  shape={df_dice.shape}")
except FileNotFoundError:
    print(f"  [INFO] DiCE 데이터 없음 — 정책 변수 생략")

# 컬럼 탐지
EMD_COL   = next((c for c in df_llm.columns if '읍면동' in c or 'emd' in c.lower()), None)
YEAR_COL  = next((c for c in df_llm.columns if '연도'   in c or c == 'year'), None)
MONTH_COL = next((c for c in df_llm.columns if c in ['사고일시_월','month']), None)
RISK_COL  = next((c for c in df_llm.columns if 'predicted_risk' in c), None)
ALERT_COL = next((c for c in df_llm.columns if 'alert' in c.lower()), None)
RISK_FACTORS = [c for c in df_llm.columns if 'risk_factor' in c]
SAFE_FACTORS = [c for c in df_llm.columns if 'safe_factor' in c]

print(f"  EMD={EMD_COL}  Year={YEAR_COL}  Month={MONTH_COL}")
print(f"  Risk={RISK_COL}  Alert={ALERT_COL}")


# =============================================================================
# 3. GPT 에이전트 클래스
# =============================================================================

class TrafficReportAgent:
    """
    GPT 기반 교통사고 위험 보고서 자동 생성 에이전트.
    SHAP 위험인자 + DiCE 반사실 + Lift 경보유형을 프롬프트에 주입하여
    구조화된 JSON 정책 보고서를 생성한다.
    """

    def __init__(self):
        self.client = OpenAI(api_key=OPENAI_API_KEY)
        self.model  = LLM_MODEL

    def _extract_factors(self, row: pd.Series) -> dict:
        risk = [feat_to_desc(str(row[c]))
                for c in RISK_FACTORS if pd.notna(row.get(c))]
        safe = [feat_to_desc(str(row[c]))
                for c in SAFE_FACTORS if pd.notna(row.get(c))]
        return {'risk': risk[:5], 'safe': safe[:5]}

    def _get_policy_vars(self) -> list:
        if not DICE_AVAILABLE or df_dice.empty:
            return []
        top = (df_dice[df_dice['Direction']=='Decrease']['Feature']
               .value_counts().head(5))
        return [{'variable': feat_to_desc(f), 'frequency': int(c), 'action': 'Reduce'}
                for f, c in top.items()]

    def generate_report(self, emd: str, year: int, month: int,
                        risk_index: float, alert_type: str,
                        factors: dict, policy_vars: list) -> dict:

        risk_level = ('HIGH'    if 'High'  in alert_type else
                      'CAUTION' if 'Over'  in alert_type else 'LOW')

        system_prompt = """
You are a traffic safety policy analyst for a South Korean local government.
Generate structured, evidence-based traffic accident risk reports
based on AI model outputs (LightGBM + SHAP + DiCE + Lift Chart).
Use formal, policy-oriented English. All output must be valid JSON.
        """

        user_prompt = f"""
[Traffic Risk Report Request]

District   : {emd} — romanized as '{emd}' (Miryang-si, South Gyeongsang Province, South Korea)
Period     : {year}-{month:02d}
Risk Index : {risk_index:.2f}  (higher = more dangerous)
Alert Level: {risk_level}  ({alert_type})

## Primary Risk Drivers (SHAP Analysis)
The following variables INCREASE accident risk (positive SHAP values).
Note: 'road_dry' means accidents occurring on dry roads — focus on BEHAVIOR
and ENFORCEMENT, not road surface removal:
{json.dumps(factors['risk'], ensure_ascii=False, indent=2)}

## Comparative Strengths (Low-risk factors)
Variables relatively safe compared to other districts:
{json.dumps(factors['safe'], ensure_ascii=False, indent=2)}

## Counterfactual Policy Variables (DiCE Analysis)
Variables that, if reduced, would shift this district to Low-Risk:
{json.dumps(policy_vars, ensure_ascii=False, indent=2)}

## Output Format (strict JSON)
{{
    "executive_summary": "2-3 sentences summarising risk level, key drivers, and urgency.",

    "risk_factor_analysis": {{
        "primary_risks": "Detailed analysis of top 3 risk factors and operational significance. 2-3 sentences.",
        "contributing_conditions": "Environmental or structural conditions amplifying risk. 1-2 sentences."
    }},

    "policy_recommendations": [
        {{
            "priority"  : 1,
            "action"    : "Specific policy measure",
            "target"    : "Target variable or behaviour",
            "rationale" : "Evidence-based justification citing AI outputs",
            "timeline"  : "Short-term (1-3 months) / Mid-term (3-12 months) / Long-term (1+ year)"
        }}
    ],

    "counterfactual_insight": "What specific changes would reduce the risk index, per DiCE. 2-3 sentences.",

    "monitoring_kpis": [
        "KPI 1: measurable indicator with numeric target (e.g., reduce X by 20% in 6 months)",
        "KPI 2: measurable indicator",
        "KPI 3: measurable indicator"
    ],

    "alert_justification": "Why this district is classified {risk_level}. Distinguish true high-risk vs over-warning if applicable."
}}

Provide at least 3 policy_recommendations ordered by priority.
        """

        try:
            resp = self.client.chat.completions.create(
                model    = self.model,
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt}
                ],
                response_format = {"type": "json_object"},
                temperature     = 0.3,
                max_tokens      = 1500,
            )
            result = json.loads(resp.choices[0].message.content)
        except Exception as e:
            result = {"error": str(e)}

        result['_meta'] = {
            'emd'          : emd,
            'year'         : year,
            'month'        : month,
            'risk_index'   : risk_index,
            'alert_type'   : alert_type,
            'model_used'   : self.model,
            'generated_at' : time.strftime('%Y-%m-%d %H:%M:%S'),
        }
        return result

    def format_text(self, report: dict) -> str:
        """JSON 보고서 → 논문 부록 / 대시보드용 텍스트 변환"""
        if 'error' in report:
            return f"[ERROR] {report['error']}"
        m = report.get('_meta', {})
        lines = [
            "=" * 65,
            f"  MIRYANG TRAFFIC RISK REPORT",
            f"  District  : {m.get('emd')}",
            f"  Period    : {m.get('year')}-{str(m.get('month')).zfill(2)}",
            f"  Risk Index: {m.get('risk_index'):.2f}  |  Alert: {m.get('alert_type')}",
            "=" * 65,
            "",
            "[ EXECUTIVE SUMMARY ]",
            report.get('executive_summary','N/A'),
            "",
            "[ RISK FACTOR ANALYSIS ]",
            "  Primary Risks:",
            f"  {report.get('risk_factor_analysis',{}).get('primary_risks','N/A')}",
            "  Contributing Conditions:",
            f"  {report.get('risk_factor_analysis',{}).get('contributing_conditions','N/A')}",
            "",
            "[ POLICY RECOMMENDATIONS ]",
        ]
        for rec in report.get('policy_recommendations', []):
            lines += [
                f"  [{rec.get('priority')}] {rec.get('action')}",
                f"      Target   : {rec.get('target')}",
                f"      Rationale: {rec.get('rationale')}",
                f"      Timeline : {rec.get('timeline')}",
                "",
            ]
        lines += [
            "[ COUNTERFACTUAL INSIGHT ]",
            report.get('counterfactual_insight','N/A'),
            "",
            "[ MONITORING KPIs ]",
        ]
        for kpi in report.get('monitoring_kpis', []):
            lines.append(f"  • {kpi}")
        lines += [
            "",
            "[ ALERT JUSTIFICATION ]",
            report.get('alert_justification','N/A'),
            "",
            f"  Generated: {m.get('generated_at')}  |  Model: {m.get('model_used')}",
            "=" * 65,
        ]
        return "\n".join(lines)


# =============================================================================
# 4. 배치 보고서 생성
# =============================================================================
print("\n" + "=" * 60)
print("STEP 2 | Generating Reports")
print("=" * 60)

agent = TrafficReportAgent()

# 고위험 읍면동 상위 5개 선택 (변경 가능)
if ALERT_COL and RISK_COL and EMD_COL:
    df_target = (df_llm[df_llm[ALERT_COL]=='High-Risk Warning']
                 .sort_values(RISK_COL, ascending=False)
                 .drop_duplicates(subset=[EMD_COL])
                 .head(5))
else:
    df_target = df_llm.head(5)

print(f"  대상 읍면동 수: {len(df_target)}")

import re

def to_int(val):
    nums = re.sub(r'[^0-9]', '', str(val))
    return int(nums) if nums else 0

def to_float(val):
    nums = re.sub(r'[^0-9.]', '', str(val))
    return float(nums) if nums else 0.0

all_reports    = []
formatted_text = []
policy_vars    = agent._get_policy_vars()

for idx, (_, row) in enumerate(df_target.iterrows()):
    emd        = str(row[EMD_COL])   if EMD_COL   else f'District_{idx+1}'
    year       = to_int(row[YEAR_COL])   if YEAR_COL  else 2022
    month      = to_int(row[MONTH_COL])  if MONTH_COL else 1
    risk_index = to_float(row[RISK_COL]) if RISK_COL  else 0.0
    alert_type = str(row[ALERT_COL]) if ALERT_COL else 'High-Risk Warning'
    factors    = agent._extract_factors(row)

    print(f"\n  [{idx+1}/{len(df_target)}] {emd} ({year}-{month:02d}) "
          f"Risk={risk_index:.2f}  {alert_type}")

    report = agent.generate_report(
        emd=emd, year=year, month=month,
        risk_index=risk_index, alert_type=alert_type,
        factors=factors, policy_vars=policy_vars
    )

    status = "✓" if 'error' not in report else f"✗ {report.get('error','')}"
    print(f"     {status}")

    all_reports.append(report)
    formatted_text.append(agent.format_text(report))
    time.sleep(1.5)


# =============================================================================
# 5. 결과 저장
# =============================================================================
print("\n" + "=" * 60)
print("STEP 3 | Saving Outputs")
print("=" * 60)

# JSON (RAG 챗봇 입력용)
with open('./traffic_reports.json', 'w', encoding='utf-8') as f:
    json.dump(all_reports, f, ensure_ascii=False, indent=2)
print("  [Saved] traffic_reports.json")

# 텍스트 (논문 부록용)
with open('./traffic_reports.txt', 'w', encoding='utf-8') as f:
    f.write("\n\n".join(formatted_text))
print("  [Saved] traffic_reports.txt")

# CSV 요약 (논문 Table VII)
flat = []
for r in all_reports:
    if 'error' in r and '_meta' not in r:
        continue
    m    = r.get('_meta', {})
    recs = r.get('policy_recommendations', [{}])
    flat.append({
        'District'              : m.get('emd'),
        'Year'                  : m.get('year'),
        'Month'                 : m.get('month'),
        'Risk Index'            : m.get('risk_index'),
        'Alert Type'            : m.get('alert_type'),
        'Executive Summary'     : r.get('executive_summary',''),
        'Primary Risks'         : r.get('risk_factor_analysis',{}).get('primary_risks',''),
        'Top Recommendation'    : recs[0].get('action','') if recs else '',
        'Counterfactual Insight': r.get('counterfactual_insight',''),
        'KPI 1'                 : (r.get('monitoring_kpis',[''])[0]
                                   if r.get('monitoring_kpis') else ''),
    })

if flat:
    pd.DataFrame(flat).to_csv('./TableVII_Report_Summary.csv',
                               index=False, encoding='utf-8-sig')
    print("  [Saved] TableVII_Report_Summary.csv")

# 샘플 출력
print("\n" + "=" * 60)
print("SAMPLE OUTPUT")
print("=" * 60)
if formatted_text:
    print(formatted_text[0])

[INFO] Model: gpt-4o-mini

STEP 1 | Loading LLM Dataset
  Loaded: ./TableVI_LLM_Dataset.csv  shape=(447, 20)
  Loaded: ./TableV_DiCE_Policy.csv  shape=(51, 6)
  EMD=시군구_읍면동명  Year=사고일시_연도  Month=사고일시_월
  Risk=predicted_risk_index  Alert=alert_type

STEP 2 | Generating Reports
  대상 읍면동 수: 5

  [1/5] 삼문동 (2021-07) Risk=192.40  High-Risk Warning
     ✓

  [2/5] 내이동 (2021-05) Risk=159.27  High-Risk Warning
     ✓

  [3/5] 상남면 (2022-08) Risk=137.97  High-Risk Warning
     ✓

  [4/5] 하남읍 (2020-06) Risk=113.50  High-Risk Warning
     ✓

  [5/5] 삼랑진읍 (2020-06) Risk=106.37  High-Risk Warning
     ✓

STEP 3 | Saving Outputs
  [Saved] traffic_reports.json
  [Saved] traffic_reports.txt
  [Saved] TableVII_Report_Summary.csv

SAMPLE OUTPUT
  MIRYANG TRAFFIC RISK REPORT
  District  : 삼문동
  Period    : 2021-07
  Risk Index: 192.40  |  Alert: High-Risk Warning

[ EXECUTIVE SUMMARY ]
The traffic accident risk index for 삼문동 stands at 192.40, categorizing it as a high-risk area. The primary drivers of thi